# M04-02 — KPIs

[← Anterior](02-lab-joins.ipynb) · [Siguiente →](04-lab-segmentacion.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Calcular GMV cobrable, nº de pedidos cobrables, ticket medio y tasa de cancelación sobre el universo **con cliente real**.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M04-02-kpis.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras. No es adorno: es la traza de tu razonamiento.
2. **Código** — lo pegas o lo escribes, lo **ejecutas** (`Shift+Enter`), **miras** la salida y, si no cuadra, lo **mejoras**.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Universo de venta

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> KPI de dinero ≠ KPI de operativa. Inner a clientes y solo is_billable para el dinero.

**2. Crea una celda de código** debajo y escribe:

```python
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col

spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
sales = (
    fact.join(customers, "customer_id", "inner")
    .where(col("is_billable"))
)
print(sales.count())
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** **1122** líneas cobrables con cliente (1127 − 5 paid huérfanas).

**Por qué este paso.** Si usas left, atribuyes GMV a CX*.


### Paso 2 — Cuatro métricas globales

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> El ticket medio se calcula a grano pedido: sum(GMV) / countDistinct(order_id), no avg de línea.

**2. Crea una celda de código** debajo y escribe:

```python
from pyspark.sql.functions import sum as fsum, countDistinct, round as fround

kpis = sales.agg(
    fround(fsum("gmv_line"), 2).alias("gmv"),
    countDistinct("order_id").alias("orders"),
)
kpis = kpis.withColumn("aov", fround(col("gmv") / col("orders"), 2))
kpis.show()
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** GMV ≈ **400157.73** · pedidos cobrables **469** · AOV ≈ **853**.

**Por qué este paso.** Si casteaste a double, el céntimo puede moverse: redondea a 2 decimales.


### Paso 3 — Tasa de cancelación

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> El denominador es pedidos (no líneas). Sobre orders_clean inner clientes (780).

**2. Crea una celda de código** debajo y escribe:

```python
from pyspark.sql.functions import avg

orders = spark.read.parquet(str(STAGING / "orders_clean"))
ord_ok = orders.join(customers, "customer_id", "inner")
cancel = ord_ok.agg(
    avg((col("status") == "cancelled").cast("double")).alias("cancel_rate")
)
cancel.show()
print("pedidos con cliente", ord_ok.count())
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** ≈ **0.22**. Pedidos con cliente **780**.

**Por qué este paso.** Si mides sobre `sales` (solo paid), la tasa sale 0.


### Paso 4 — KPI por canal

**1. Crea una celda Markdown** en *tu* notebook. Explica con tus palabras (puedes partir de esto):

> channel_norm (no channel) evita partir web/WEB. Ordeno por GMV.

**2. Crea una celda de código** debajo y escribe:

```python
(
    sales.groupBy("channel_norm")
    .agg(
        fround(fsum("gmv_line"), 2).alias("gmv"),
        countDistinct("order_id").alias("orders"),
    )
    .orderBy(col("gmv").desc())
    .show()
)
```

**3. Ejecuta** esa celda (`Shift+Enter`). Espera a que deje de verse `[*]`.

**4. Comprueba.** Cuatro filas (`app`, `other`, `store`, `web`). `web` o `app` en cabeza.

**Por qué este paso.** Este groupBy es el cuadro de mando.


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Reproduce `gmv / countDistinct(order_id)` solo con is_billable e inner.
Un número ~850, no ~350 (eso sería media de línea). Escríbelo en Markdown.


## Mejora — GMV por mes y país

`groupBy("order_month", "country")` con la misma regla cobrable. `UNK` aparece si no rellenaste país.

<details>
<summary>Si te atascas, mira una solución</summary>

```python
(
    sales.groupBy("order_month", "country")
    .agg(fround(fsum("gmv_line"), 2).alias("gmv"))
    .orderBy("order_month", "country")
    .show(20)
)
```

</details>


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| GMV ~ 2× | Join al catálogo duplicado | `dropDuplicates(["product_id"])` |
| AOV ridículamente bajo | `avg("gmv_line")` | `sum / countDistinct(order_id)` |
| Cancel rate 0 | Mediste sobre sales (solo paid) | Usa orders_clean |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M04-03 segmentación](04-lab-segmentacion.ipynb).
